In [ ]:
# ==============================
# TASK 4: Forecasting (One Cell)
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# ------------------------------
# Load Dataset
# ------------------------------
df = pd.read_csv("../data/processed/ethiopia_fi_enriched.csv")   # Change path if needed

# ------------------------------
# Function for Forecasting
# ------------------------------
def forecast_indicator(df, indicator_code, title):
    
    data = df[
        (df["record_type"]=="observation") &
        (df["indicator_code"]==indicator_code)
    ].copy()

    data["year"] = pd.to_datetime(data["observation_date"]).dt.year
    data = data.sort_values("year")

    X = data[["year"]]
    y = data["value_numeric"]

    # Train Model
    model = LinearRegression()
    model.fit(X, y)

    # Historical Predictions
    fitted = model.predict(X)

    # Future Years
    future = pd.DataFrame({
        "year":[2025,2026,2027]
    })

    baseline = model.predict(future)

    # Residual Standard Error
    residuals = y - fitted
    std = residuals.std()

    # Confidence Interval
    lower = baseline - 1.96*std
    upper = baseline + 1.96*std

    # Scenario Forecasts
    optimistic = baseline + std
    pessimistic = baseline - std

    forecast = pd.DataFrame({
        "Year":future["year"],
        "Baseline":baseline.round(2),
        "Lower95CI":lower.round(2),
        "Upper95CI":upper.round(2),
        "Optimistic":optimistic.round(2),
        "Pessimistic":pessimistic.round(2)
    })

    print("="*60)
    print(title)
    print("="*60)
    print(forecast)

    # Plot
    plt.figure(figsize=(10,5))

    plt.plot(
        data["year"],
        y,
        marker="o",
        linewidth=2,
        label="Historical"
    )

    plt.plot(
        future["year"],
        baseline,
        marker="o",
        linestyle="--",
        linewidth=2,
        label="Baseline Forecast"
    )

    plt.fill_between(
        future["year"],
        lower,
        upper,
        alpha=0.3,
        label="95% Confidence Interval"
    )

    plt.plot(
        future["year"],
        optimistic,
        linestyle=":",
        linewidth=2,
        label="Optimistic"
    )

    plt.plot(
        future["year"],
        pessimistic,
        linestyle=":",
        linewidth=2,
        label="Pessimistic"
    )

    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel("Value (%)")
    plt.grid(True)
    plt.legend()
    plt.show()

    return forecast

# ------------------------------
# Account Ownership Forecast
# ------------------------------
account_forecast = forecast_indicator(
    df,
    "ACC_OWNERSHIP",
    "Account Ownership Forecast (2025–2027)"
)

# ------------------------------
# Digital Payment Usage Forecast
# ------------------------------
digital_forecast = forecast_indicator(
    df,
    "USG_DIGITAL_PAYMENT",
    "Digital Payment Usage Forecast (2025–2027)"
)

print("\nForecasting Complete!")